In [1]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

## L_CA1 — CA1 Field

**Role**: Convergence of MSP (ECin → CA1, slow) and TSP (CA3 → CA1, fast).
The plus-phase ECout back-projection corrects CA1 toward the target item.

**Key parameters (Schapiro 2017 §2.a.iv / SI Table 1)**:
- `W_ECin`: MSP pathway — fully connected; scale=3 (SI Table 2); lr=0.05 (slow → community statistics)
- `W_CA3`: TSP pathway — fully connected; lr=0.4 (fast → episodic binding)
- `W_ECout`: back-projection — active in both minus (free) and plus (clamped) phases
- `k_frac = 0.25`: ~25% active (SI Table 1; much less sparse than DG/CA3)

**Phase structure (Schapiro 2017 §2.b)**:
- Q1 (cycles 1–25): pass `a_CA3=zeros` → ECin-dominant (theta trough; encoding)
- Q2–Q3 (cycles 26–75): pass `a_ECin=zeros` → CA3-dominant (theta peak; retrieval)
- Q4 (cycles 76–100): pass `a_ECout=target` → plus phase (correction)

**Understanding check**: Why does the learning rate for W_ECin need to be 10× slower than W_CA3?  
→ MSP must accumulate statistics across many trials to reflect community structure.  
If lr_MSP were as fast as lr_TSP, each trial would overwrite previous statistical patterns.  
TSP needs one-shot binding; if too slow, the episode is lost before W_CA3 strengthens.

In [ ]:
# path & directories
import sys
from pathlib import Path

DIR_SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
DIR_VIZ = (Path(__vsc_ipynb_file__).parent.parent / 'visualizations').resolve()
sys.path.insert(0, DIR_SRC)

# upstream activations — DG and CA3 outputs used by CA1 phase tests below
import torch
from layer import L_DG, L_CA3, L_CA1

# ECin: item A (unit 0 = current at 1.0; unit 1 = previous at 0.9)
# Schapiro 2017 §2.c: moving-window encoding of current and prior item
a_ECin_A = torch.zeros(15); a_ECin_A[0] = 1.0; a_ECin_A[1] = 0.9

# DG — default uniform(0.25, 0.75) weights; settle 25 cycles (k_frac=0.01 → ~1 active unit)
torch.manual_seed(42)
dg = L_DG(n_input=15, n_DG=100, k_frac=0.01, ecin_frac=0.25)
dg.reset()
for _ in range(25):
    act_DG_A = dg(a_ECin_A)
act_DG_A = act_DG_A.clone()

# CA3 — default uniform(0.25, 0.75) weights; settle 25 cycles (k_frac=0.06 → ~3 active units)
torch.manual_seed(0)
ca3 = L_CA3(n_DG=100, n_ECin=15, n_CA3=50, k_frac=0.06, dg_frac=0.05)
ca3.reset()
for _ in range(25):
    act_CA3_A = ca3(act_DG_A, a_ECin_A)
act_CA3_A = act_CA3_A.clone()

print(f"DG  active: {(act_DG_A  > 0).sum().item()} / 100  (expect ~1,  k_frac=0.01)")
print(f"CA3 active: {(act_CA3_A > 0).sum().item()} / 50   (expect ~3,  k_frac=0.06)")

In [ ]:
# GOAL: Confirm L_CA1 has W_ECin (MSP), W_CA3 (TSP), W_ECout (back-proj) with correct shapes
# Instantiate and inspect structure
# Expected:
#   W_ECin  shape: torch.Size([15, 50]) — MSP; ECin → CA1 (Schapiro 2017 §2.a.iv)
#   W_CA3   shape: torch.Size([50, 50]) — TSP; CA3 → CA1
#   W_ECout shape: torch.Size([15, 50]) — back-projection; ECout → CA1 (plus-phase teaching)
#   n_active target: 12                 — 25% of 50 CA1 units (SI Table 1)
ca1 = L_CA1(n_items=15, n_CA3=50, n_CA1=50, k_frac=0.25, use_euler=True)

print(f"W_ECin  shape: {ca1.W_ECin.shape}")
print(f"W_CA3   shape: {ca1.W_CA3.shape}")
print(f"W_ECout shape: {ca1.W_ECout.shape}")
print(f"n_active target: {max(1, int(ca1.k_frac * ca1.n_CA1))}")

In [ ]:
# GOAL: Confirm ECin-dominant (Q1) and CA3-dominant (Q2-Q3) CA1 patterns differ in isolation
# Phase switch: Q1 (ECin-dominant) vs Q2–Q3 (CA3-dominant)
# Theta trough → ECin drives CA1; theta peak → CA3 drives CA1 (Schapiro 2017 §2.b).
# Passing zeros for the inactive pathway implements theta_discrete convention.
# Default uniform(0.25, 0.75) weights; 25/50-cycle settling ensures non-zero activity.
#
# Note: reset() is called between Q1 and Q2–Q3 here intentionally.
# This tests each pathway IN ISOLATION to confirm they produce different CA1 patterns.
# In a real trial (M_Hip), there is NO reset between Q1 and Q2–Q3 — Q2–Q3 continues
# from Q1's final Euler state, so CA3 pulls the activity away from Q1's pattern rather
# than starting fresh. That continuous dynamics test belongs in test_M_Hip.ipynb.
#
# Expected:
#   Q1  active : ~12 / 50  — ECin drives 25% of CA1 units
#   Q23 active : ~12 / 50  — CA3 drives 25% of CA1 units (same sparsity, different units)
#   Cosine(Q1, Q2-3) < 1.0 — two different pathways produce different CA1 patterns

# Q1 — ECin active, CA3 zeroed (theta trough; encoding; 25 cycles)
ca1.reset()
for _ in range(25):
    act_q1 = ca1(a_ECin_A, torch.zeros(50))
act_q1 = act_q1.clone()

# Q2–Q3 — ECin zeroed, CA3 active (theta peak; retrieval; 50 cycles)
# reset() here is for test isolation only — not part of real trial structure
ca1.reset()
for _ in range(50):
    act_q23 = ca1(torch.zeros(15), act_CA3_A)
act_q23 = act_q23.clone()

cos = torch.nn.functional.cosine_similarity(act_q1.unsqueeze(0), act_q23.unsqueeze(0)).item()
print(f"Q1  active: {(act_q1  > 0).sum().item()} / {ca1.n_CA1}")
print(f"Q23 active: {(act_q23 > 0).sum().item()} / {ca1.n_CA1}")
print(f"Cosine(Q1, Q2-3): {cos:.3f}  (different pathways → different CA1 patterns)")

In [ ]:
# GOAL: Confirm ECout back-projection shifts CA1 away from minus-phase state
# Plus phase: ECout back-projection shifts CA1 activity toward the target item.
# Q4 continues from Q2–Q3 final state — no reset between phases (Schapiro 2017 §2.b).
#
# To isolate ECout's effect, Q4 is run twice from the same minus-phase endpoint:
#   - WITHOUT ECout → ActP_free    (ECout=zeros; CA1 settles on its own)
#   - WITH    ECout → ActP_clamped (ECout clamped to item 3)
#
# Expected:
#   cosine(ActM, ActP_free)    = 1.000  — no back-projection → CA1 identical to ActM
#   cosine(ActM, ActP_clamped) < 1.000  — ECout fires → some CA1 units switch
#
# Why is the shift small (cosine close to 1.0) with untrained W_ECout?
#   net_CA1 = W_ECin @ a_ECin + W_CA3 @ a_CA3 + W_ECout @ a_ecout_target
#   a_ecout_target has only unit 3 = 1.0, so ECout contributes W_ECout[3, :] only.
#   Untrained W_ECout[3, :] is random (uniform 0.25–0.75), adding roughly equal
#   drive to ALL CA1 units. Equal push → kWTA ranking barely changes → small shift.
#   After training, W_ECout[3, :] is strong only for units active in item 3's CA1
#   pattern → targeted drive → larger shift → CA1 pulled toward the correct pattern.
a_ecout_target = torch.zeros(15); a_ecout_target[3] = 1.0   # next item = item 3

# Q2–Q3 minus phase
ca1.reset()
for _ in range(50):
    act_minus_end = ca1(a_ECin_A, act_CA3_A)
act_minus_end = act_minus_end.clone()

# save minus-phase Euler state to replay Q4 twice from the same starting point
Vm_saved = ca1._Vm.data.clone()
y_saved  = ca1._y.data.clone()

# Q4 without ECout
for _ in range(25):
    act_plus_free = ca1(a_ECin_A, act_CA3_A)
act_plus_free = act_plus_free.clone()

# restore state and run Q4 with ECout clamped
ca1._Vm.data.copy_(Vm_saved)
ca1._y.data.copy_(y_saved)
for _ in range(25):
    act_plus_clamped = ca1(a_ECin_A, act_CA3_A, a_ECout=a_ecout_target)
act_plus_clamped = act_plus_clamped.clone()

cos_free    = torch.nn.functional.cosine_similarity(act_minus_end.unsqueeze(0), act_plus_free.unsqueeze(0)).item()
cos_clamped = torch.nn.functional.cosine_similarity(act_minus_end.unsqueeze(0), act_plus_clamped.unsqueeze(0)).item()

print(f"cosine(ActM, ActP_free)    = {cos_free:.3f}  (no ECout  → CA1 unchanged)")
print(f"cosine(ActM, ActP_clamped) = {cos_clamped:.3f}  (ECout=item3 → CA1 shifted)")

In [ ]:
# GOAL: Confirm W_CA3 update is ~8× W_ECin update (lr_TSP=0.4 vs lr_MSP=0.05)
# Sanity check: not a novel test — confirms update_weights applies learning rates correctly.
# CHL lr ratio: W_CA3 / W_ECin update ≈ 8× (lr_TSP / lr_MSP = 0.4 / 0.05).
# Also confirms W_ECout uses full CHL (both minus and plus terms contribute).
#
# Expected:
#   max |ΔW_ECin | = small           — lr_MSP=0.05; MSP learns slowly
#   max |ΔW_CA3  | ≈ 8× larger      — lr_TSP=0.4; TSP is 8× faster
#   max |ΔW_ECout| > 0               — full CHL: minus and plus ECout both contribute
#   Ratio ΔW_CA3 / ΔW_ECin ≈ 8×
ca1_test = L_CA1(n_items=15, n_CA3=50, n_CA1=50, k_frac=0.25, lr_MSP=0.05, lr_TSP=0.4)

ca1_test.reset()
for _ in range(50):
    a_ca1_m = ca1_test(a_ECin_A, act_CA3_A)
a_ca1_m = a_ca1_m.clone()

for _ in range(25):
    a_ca1_p = ca1_test(a_ECin_A, act_CA3_A, a_ECout=a_ecout_target)
a_ca1_p = a_ca1_p.clone()

a_ecout_m = torch.rand(15) * 0.1   # simulate free minus-phase ECout activity

W_ECin_before  = ca1_test.W_ECin.data.clone()
W_CA3_before   = ca1_test.W_CA3.data.clone()
W_ECout_before = ca1_test.W_ECout.data.clone()

ca1_test.update_weights(
    a_ECin=a_ECin_A,
    a_CA3_minus=act_CA3_A,    a_CA3_plus=act_CA3_A,
    a_ECout_minus=a_ecout_m,  a_ECout_plus=a_ecout_target,
    a_CA1_minus=a_ca1_m,      a_CA1_plus=a_ca1_p,
)

dW_ECin  = (ca1_test.W_ECin.data  - W_ECin_before).abs().max().item()
dW_CA3   = (ca1_test.W_CA3.data   - W_CA3_before).abs().max().item()
dW_ECout = (ca1_test.W_ECout.data - W_ECout_before).abs().max().item()

print(f"max |ΔW_ECin | = {dW_ECin:.5f}  (lr_MSP={ca1_test.lr_MSP})")
print(f"max |ΔW_CA3  | = {dW_CA3:.5f}  (lr_TSP={ca1_test.lr_TSP})")
print(f"max |ΔW_ECout| = {dW_ECout:.5f}  (lr_MSP={ca1_test.lr_MSP}, full CHL)")
print(f"Ratio ΔW_CA3 / ΔW_ECin ≈ {dW_CA3 / (dW_ECin + 1e-9):.1f}  (expect ~{ca1_test.lr_TSP/ca1_test.lr_MSP:.0f}×)")